# Load Model

In [1]:
import tensorflow as tf
import numpy as np

# ==========================================
# 1. HIDE THE GPU (CRITICAL FIX FOR CUDNN)
# ==========================================
tf.config.set_visible_devices([], 'GPU')

print("Loading model...")
model = tf.keras.models.load_model('/kaggle/input/models/basanttyasser/violencemodel/keras/default/1/modelv2.keras')

2026-04-12 11:29:49.690710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775993389.872067      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775993389.926223      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775993390.365196      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775993390.365241      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775993390.365244      55 computation_placer.cc:177] computation placer alr

Loading model...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 104 variables whereas the saved optimizer has 108 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## A: Dynamic-Range-Quantization

In [3]:
# 2. Initialize converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 3. Add the flags to prevent the converter from hanging / failing on the LSTM
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # Standard TFLite ops
    tf.lite.OpsSet.SELECT_TF_OPS    # Let it use TF ops for the complex LSTM parts
]
converter._experimental_lower_tensor_list_ops = False

# 4. Convert (This should be much faster!)
print("Starting conversion... this may still take 2-5 minutes, please wait.")
tflite_quant_model = converter.convert()

# 5. Save the file
with open('model_dynamic_quant.tflite', 'wb') as f:
    f.write(tflite_quant_model)
    
print("Quantized model saved successfully!")

Starting conversion... this may still take 2-5 minutes, please wait.


KeyboardInterrupt: 

## B: Float16-Quantization

In [2]:
# 3. Initialize converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 4. Set to Float16 (Method B)
converter.target_spec.supported_types = [tf.float16]

# 5. Fix for dynamic LSTM loops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, 
    tf.lite.OpsSet.SELECT_TF_OPS    
]
converter._experimental_lower_tensor_list_ops = False

print("Starting conversion... please wait. This may take a few minutes.")

# 6. Convert the model
tflite_fp16_model = converter.convert()

# 7. Save the file
with open('model_fp16_quant.tflite', 'wb') as f:
    f.write(tflite_fp16_model)
    
print("Float16 Quantized model saved successfully!")

Starting conversion... please wait. This may take a few minutes.
INFO:tensorflow:Assets written to: /tmp/tmp6cqcvgmr/assets


INFO:tensorflow:Assets written to: /tmp/tmp6cqcvgmr/assets


Saved artifact at '/tmp/tmp6cqcvgmr'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 16, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  134557616995216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612229648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612229840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557616996176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612229072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612230416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612231376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612230800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612230992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612231952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134557612

W0000 00:00:1775992887.465960      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1775992887.466041      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1775992888.143179      55 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Float16 Quantized model saved successfully!


## C: Full-Intger-Quantization

In [2]:
# ==========================================
# 2. CREATE A REPRESENTATIVE DATASET
# ==========================================
# TFLite needs to push data through the model to calibrate INT8 ranges.
# IMPORTANT: Replace `dummy_data` below with a real slice of your training video data!
def representative_data_gen():
    # We run 100 samples through the model for calibration
    for _ in range(100):
        # Based on your logs, your model expects: (Batch=1, Frames=16, H=224, W=224, Channels=3)
        # Ideally, load 1 real video array here instead of random numbers.
        dummy_data = np.random.rand(1, 16, 224, 224, 3).astype(np.float32)
        yield [dummy_data]

# ==========================================
# 3. FORCE A STATIC SHAPE (Crucial for INT8 LSTMs)
# ==========================================
# We replace the dynamic "None" batch size with a strict "1"
run_model = tf.function(lambda x: model(x))
concrete_func = run_model.get_concrete_function(
    tf.TensorSpec(shape=(1, 16, 224, 224, 3), dtype=tf.float32)
)

# Initialize the converter using the locked-shape function
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# ==========================================
# 4. INT8 & LSTM FLAGS
# ==========================================
# Provide the dataset generator
converter.representative_dataset = representative_data_gen

# Set supported ops to allow INT8, standard ops, AND fallback TF ops for the LSTM loops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8, # Force INT8 where possible (CNN parts)
    tf.lite.OpsSet.TFLITE_BUILTINS,      # Fallback to Float32 standard ops if needed
    tf.lite.OpsSet.SELECT_TF_OPS         # Fallback to TF ops for LSTM loops
]
converter._experimental_lower_tensor_list_ops = False

# Force the Input and Output arrays to be purely INT8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# ==========================================
# 5. CONVERT AND SAVE
# ==========================================
print("Starting Full INT8 Quantization... (This will take a few minutes)")
tflite_int8_model = converter.convert()

with open('model_full_int8.tflite', 'wb') as f:
    f.write(tflite_int8_model)
    
print("Full INT8 Quantized model saved successfully!")

I0000 00:00:1775993433.934218      55 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1


Starting Full INT8 Quantization... (This will take a few minutes)


I0000 00:00:1775993433.934488      55 single_machine.cc:374] Starting new session
I0000 00:00:1775993434.038307      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1775993448.507319      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1775993448.507369      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite delegate for select TF ops.
I0000 00:00:1775993451.281622      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Full INT8 Quantized model saved successfully!
